# データ可視化
- CSV➡NPZ、配列データに変換
  - 可視化
- パッチとして切り出し
- Augmentation 
- 最終、ptとして保存。
- 最後、モデルを訓練させるところまでコードを実装しなおした。

In [ ]:
import pandas as pd
dir = "../data/processed/20250825_sample"

Q_1_path = "heater_out_1.csv"
Q_2_path = "heater_out_2.csv"
T_1_path = "sera_temp_1.csv"
T_2_path = "sera_temp_2.csv"

Q_1 = pd.read_csv(dir + "/" + Q_1_path)
Q_2 = pd.read_csv(dir + "/" + Q_2_path)
T_1 = pd.read_csv(dir + "/" + T_1_path)
T_2 = pd.read_csv(dir + "/" + T_2_path)



In [12]:
import os
import numpy as np
import matplotlib.pyplot as plt

# バージョン0の出力先（ノートブックからの相対パス）
OUT_DIR = "../saved/visualized"


def _build_xy_grid(df: pd.DataFrame, value_col: str):
    agg = (
        df.groupby(["Y", "X"], as_index=False)[value_col]
        .mean()
        .sort_values(["Y", "X"])
    )
    xs = np.unique(agg["X"].to_numpy())
    ys = np.unique(agg["Y"].to_numpy())
    grid = np.full((ys.size, xs.size), np.nan, dtype=float)
    xi = {v: i for i, v in enumerate(xs)}
    yi = {v: i for i, v in enumerate(ys)}
    for _, row in agg.iterrows():
        grid[yi[row["Y"]], xi[row["X"]]] = row[value_col]
    return xs, ys, grid


def _edges(vals: np.ndarray) -> np.ndarray:
    if vals.size == 1:
        v = vals[0]
        return np.array([v - 0.5, v + 0.5])
    diffs = np.diff(vals)
    step = np.median(diffs)
    start = vals[0] - step / 2
    end = vals[-1] + step / 2
    return np.linspace(start, end, vals.size + 1)


def _remap_grid(xs_src, ys_src, grid_src, xs_dst, ys_dst):
    xi = {v: i for i, v in enumerate(xs_src)}
    yi = {v: i for i, v in enumerate(ys_src)}
    out = np.full((ys_dst.size, xs_dst.size), np.nan, dtype=float)
    for j, y in enumerate(ys_dst):
        if y not in yi:
            continue
        for i, x in enumerate(xs_dst):
            if x not in xi:
                continue
            out[j, i] = grid_src[yi[y], xi[x]]
    return out


def _align_two_grids(xs_a, ys_a, g_a, xs_b, ys_b, g_b):
    xs = xs_a if np.array_equal(xs_a, xs_b) else np.intersect1d(xs_a, xs_b)
    ys = ys_a if np.array_equal(ys_a, ys_b) else np.intersect1d(ys_a, ys_b)
    if not (np.array_equal(xs, xs_a) and np.array_equal(ys, ys_a)):
        g_a = _remap_grid(xs_a, ys_a, g_a, xs, ys)
    if not (np.array_equal(xs, xs_b) and np.array_equal(ys, ys_b)):
        g_b = _remap_grid(xs_b, ys_b, g_b, xs, ys)
    return xs, ys, g_a, g_b


def _load_xy_value(csv_path: str, value_candidates: tuple[str, ...]) -> tuple[pd.DataFrame, str]:
    if not os.path.isfile(csv_path):
        raise FileNotFoundError(csv_path)
    df = pd.read_csv(csv_path)
    for c in value_candidates:
        if c in df.columns:
            return df.dropna(subset=["X", "Y", "Z"]), c
    raise ValueError(f"想定列が見つかりません {value_candidates}。実際の列: {list(df.columns)}")


def _save_heatmap(df: pd.DataFrame, value_col: str, title: str, out_path: str, cmap: str) -> None:
    os.makedirs(os.path.dirname(os.path.abspath(out_path)) or ".", exist_ok=True)
    xs, ys, grid = _build_xy_grid(df, value_col)
    x_edges, y_edges = _edges(xs), _edges(ys)
    fig, ax = plt.subplots(figsize=(7, 6))
    mesh = ax.pcolormesh(
        x_edges,
        y_edges,
        np.ma.masked_invalid(grid),
        cmap=cmap,
        shading="auto",
    )
    plt.colorbar(mesh, ax=ax, label=value_col)
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_title(title)
    ax.set_aspect("equal", adjustable="box")
    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.close(fig)


def save_q_visualization(csv_path: str, out_name: str) -> None:
    """発熱（Q）単体。カラーマップは viridis。"""
    df, col = _load_xy_value(csv_path, ("Q_1", "Q_2", "heater_output_W_m-3"))
    _save_heatmap(df, col, f"Q", out_name, cmap="viridis")


def save_t_visualization(csv_path: str, out_name: str) -> None:
    """温度（T）単体。カラーマップは jet。"""
    df, col = _load_xy_value(csv_path, ("T_1", "T_2", "temperature_degC"))
    _save_heatmap(df, col, f"T", out_name, cmap="jet")


def save_q_t_pair_visualization(q_csv_path: str, t_csv_path: str, out_name: str) -> None:
    """Q と T を横並びで同一 XY 軸に揃えて保存。Q=viridis、T=jet。"""
    df_q, col_q = _load_xy_value(q_csv_path, ("Q_1", "Q_2", "heater_output_W_m-3"))
    df_t, col_t = _load_xy_value(t_csv_path, ("T_1", "T_2", "temperature_degC"))
    xs_q, ys_q, grid_q = _build_xy_grid(df_q, col_q)
    xs_t, ys_t, grid_t = _build_xy_grid(df_t, col_t)
    xs, ys, grid_q, grid_t = _align_two_grids(xs_q, ys_q, grid_q, xs_t, ys_t, grid_t)
    x_edges, y_edges = _edges(xs), _edges(ys)

    os.makedirs(os.path.dirname(os.path.abspath(out_name)) or ".", exist_ok=True)
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)
    for ax, grid, cmap, label, ttl in [
        (axes[0], grid_q, "viridis", col_q, f"Q"),
        (axes[1], grid_t, "jet", col_t, f"T"),
    ]:
        mesh = ax.pcolormesh(
            x_edges, y_edges, np.ma.masked_invalid(grid), cmap=cmap, shading="auto"
        )
        fig.colorbar(mesh, ax=ax, label=label, fraction=0.046, pad=0.04)
        ax.set_xlabel("X")
        ax.set_ylabel("Y")
        ax.set_title(ttl)
        ax.set_aspect("equal", adjustable="box")
    fig.suptitle("Q | T (pair)")
    fig.tight_layout()
    fig.savefig(out_name, dpi=200)
    plt.close(fig)


def save_q_t_diff_visualization(
    q_csv_1: str,
    q_csv_2: str,
    t_csv_1: str,
    t_csv_2: str,
    out_name: str,
) -> None:
    """Q を2本・T を2本渡し、同じ XY 上で (2番目 - 1番目) の差分を左右に描画。カラーマップは coolwarm（0 基準）。"""
    df_q1, c_q1 = _load_xy_value(q_csv_1, ("Q_1", "Q_2", "heater_output_W_m-3"))
    df_q2, c_q2 = _load_xy_value(q_csv_2, ("Q_1", "Q_2", "heater_output_W_m-3"))
    df_t1, c_t1 = _load_xy_value(t_csv_1, ("T_1", "T_2", "temperature_degC"))
    df_t2, c_t2 = _load_xy_value(t_csv_2, ("T_1", "T_2", "temperature_degC"))

    xs_q1, ys_q1, g_q1 = _build_xy_grid(df_q1, c_q1)
    xs_q2, ys_q2, g_q2 = _build_xy_grid(df_q2, c_q2)
    xs_t1, ys_t1, g_t1 = _build_xy_grid(df_t1, c_t1)
    xs_t2, ys_t2, g_t2 = _build_xy_grid(df_t2, c_t2)

    xs_q, ys_q, g_q1a, g_q2a = _align_two_grids(xs_q1, ys_q1, g_q1, xs_q2, ys_q2, g_q2)
    xs_t, ys_t, g_t1a, g_t2a = _align_two_grids(xs_t1, ys_t1, g_t1, xs_t2, ys_t2, g_t2)

    def diff_with_nan_rules(a, b):
        both_nan = np.isnan(a) & np.isnan(b)
        aa = np.where(np.isnan(a), 0.0, a)
        bb = np.where(np.isnan(b), 0.0, b)
        d = aa - bb
        d[both_nan] = np.nan
        return d

    diff_q = diff_with_nan_rules(g_q2a, g_q1a)
    diff_t = diff_with_nan_rules(g_t2a, g_t1a)

    vmax_q = np.nanmax(np.abs(diff_q))
    vmax_t = np.nanmax(np.abs(diff_t))
    if not np.isfinite(vmax_q) or vmax_q == 0:
        vmax_q = 1.0
    if not np.isfinite(vmax_t) or vmax_t == 0:
        vmax_t = 1.0

    os.makedirs(os.path.dirname(os.path.abspath(out_name)) or ".", exist_ok=True)
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=False, sharey=False)
    for ax, diff, xe, ye, title, vmax in [
        (axes[0], diff_q, _edges(xs_q), _edges(ys_q), f"Diff Q", vmax_q),
        (axes[1], diff_t, _edges(xs_t), _edges(ys_t), f"Diff T", vmax_t),
    ]:
        mesh = ax.pcolormesh(
            xe,
            ye,
            np.ma.masked_invalid(diff),
            cmap="coolwarm",
            shading="auto",
            vmin=-vmax,
            vmax=vmax,
        )
        fig.colorbar(mesh, ax=ax, fraction=0.046, pad=0.04)
        ax.set_xlabel("X")
        ax.set_ylabel("Y")
        ax.set_title(title)
        ax.set_aspect("equal", adjustable="box")
    fig.tight_layout()
    fig.savefig(out_name, dpi=200)
    plt.close(fig)


# --- バージョン0の出力例（0 = 可視化ロジックのバージョン番号） ---
os.makedirs(OUT_DIR, exist_ok=True)
save_q_visualization(os.path.join(dir, Q_1_path), os.path.join(OUT_DIR, "heater_out_0_1.png"))
save_t_visualization(os.path.join(dir, T_1_path), os.path.join(OUT_DIR, "sera_temp_0_1.png"))
save_q_t_pair_visualization(
    os.path.join(dir, Q_1_path),
    os.path.join(dir, T_1_path),
    os.path.join(OUT_DIR, "pair_Q_T_0_1.png"),
)
save_q_t_diff_visualization(
    os.path.join(dir, Q_1_path),
    os.path.join(dir, Q_2_path),
    os.path.join(dir, T_1_path),
    os.path.join(dir, T_2_path),
    os.path.join(OUT_DIR, "diff_Q_T_0_1.png"),
)
save_q_t_pair_visualization(
    os.path.join(dir, Q_2_path),
    os.path.join(dir, T_2_path),
    os.path.join(OUT_DIR, "pair_Q_T_0_2.png"),
)

<img src="pair_Q_T_0_1.png">
<img src="pair_Q_T_0_2.png">

## データ準備〜学習（処理の流れ）
上から順番に追ってくことで、学習まで進めた。
- `csv2npz.py` … CSV をフルグリッド NPZ に変換（同名 `.npz`）。
- `npz2augmentation.py` … フル NPZ をスライド窓でパッチ化し、回転・反転の Augmentation 付きで `Q_patches` / `T_patches` に保存。
- `ml_data_preparation.py` … Q–T のパッチを座標で対応づけ、train/val/test 分割・`StandardScaler` 適用後に `.pt` と `q_scaler.pkl` / `t_scaler.pkl` 等を保存。
- `unet_train.py` … 単発の U-Net 学習（MSE・W&B ログ）。※必要なら。
- `optuna_unet.py` … バッチサイズや学習率などを Optuna で探索し、Trial ごとにベストモデル等を `Trial0/`… に保存、`result.json` に集約。


## 評価はまだ。